# Grover's Algorithm — 4 Qubits
### Marked state: |0010⟩

**Refactored for Qiskit 1.x** — original notebook: `Grovers_algo-4_qubits.ipynb`

> Reference: [Figgatt et al. / Grover's search via trapped-ion gates](http://www.diva-portal.org/smash/get/diva2:1214481/FULLTEXT01.pdf)

---
## Background

**Grover's algorithm** solves the *unstructured search problem*: given a black-box function $f:\{0,1\}^n \to \{0,1\}$ that marks exactly one input, find the marked input using $O(\sqrt{N})$ queries — a quadratic speedup over classical $O(N)$.

For $n=4$ qubits, $N = 2^4 = 16$ states. The optimal number of Grover iterations is
$$k = \left\lfloor \frac{\pi}{4}\sqrt{N} \right\rfloor = \left\lfloor \frac{\pi}{4} \times 4 \right\rfloor = 3$$

### Each Grover iteration has two stages

1. **Oracle** $\hat{O}$: flips the phase of the marked state $|\omega\rangle$ via $|x\rangle \mapsto (-1)^{f(x)}|x\rangle$
2. **Diffusion** (Grover diffuser) $\hat{D} = 2|s\rangle\langle s| - I$: reflects about the uniform superposition, amplifying the marked amplitude

### Circuit implementation

The oracle for $|\omega\rangle = |0010\rangle$ is constructed by:
- Applying $X$ gates to qubits that are 0 in $|\omega\rangle$ (qubits 0, 2, 3) — converting 0→1 so all qubits are 1 only for the marked state
- Applying a 4-qubit controlled-$Z$ (phase flip when all qubits are $|1\rangle$)
- Undoing the $X$ gates

The diffuser is $H \cdot X \cdot \text{cccZ} \cdot X \cdot H$.

---
## 1. Imports

**Qiskit 1.x changes from the original:**
- `qiskit_aer` is now a separate package — `AerSimulator` replaces `BasicAer`'s `qasm_simulator`
- `execute()` is removed — use `backend.run(transpile(circuit, backend))`
- `IBMQ` (legacy IBM Quantum provider) is removed from open-source Qiskit
- `cu1` gate renamed to `cp` (controlled-phase)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
import numpy as np
from math import pi

# Qiskit 1.x core
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit.compiler import transpile

# Aer simulator (separate package: pip install qiskit-aer)
from qiskit_aer import AerSimulator

# Visualisation
from qiskit.visualization import plot_histogram

---
## 2. Multi-controlled Z gate

The oracle and diffuser both require a **4-qubit controlled-Z** (3 controls + 1 target). Qiskit does not natively decompose `mcz` into basis gates for arbitrary depth, so we implement it explicitly.

The function below handles 1, 2, or 3 control qubits:

| Controls | Decomposition |
|---|---|
| 1 | $H \cdot \text{CNOT} \cdot H$ (CZ via conjugation) |
| 2 | $H \cdot \text{CCX} \cdot H$ (Toffoli-based CZ) |
| 3 | Phase-gate ladder (Selinger / standard) |

The 3-control case uses the **$T$-gate decomposition** of the Toffoli, implemented via `cp` (controlled-phase, formerly `cu1`) gates — the key original contribution of this notebook.

> **Qiskit 1.x note:** `cu1(θ, a, b)` is now `cp(θ, a, b)`. The gate is identical — controlled-phase $\begin{pmatrix}1&0\\0&e^{i\theta}\end{pmatrix}$.

In [ ]:
def n_controlled_Z(circuit, controls, target):
    """Implement a Z gate with multiple controls (1, 2, or 3).

    Uses H-conjugation for 1/2 controls and the phase-gate ladder
    decomposition for 3 controls.  The 3-control case is built from
    cp (controlled-phase) gates — equivalent to the T-gate decomposition
    of the CCX but applied in the phase basis.
    """
    if len(controls) > 3:
        raise ValueError('The controlled-Z with more than 3 controls is not implemented')

    elif len(controls) == 1:
        circuit.h(target)
        circuit.cx(controls[0], target)
        circuit.h(target)

    elif len(controls) == 2:
        circuit.h(target)
        circuit.ccx(controls[0], controls[1], target)
        circuit.h(target)

    elif len(controls) == 3:
        # Phase-gate ladder decomposition of the 3-controlled Z
        # (cp is the Qiskit 1.x name for cu1 — controlled-phase gate)
        circuit.cp(pi/4,  controls[0], target)
        circuit.cx(controls[0], controls[1])
        circuit.cp(-pi/4, controls[1], target)
        circuit.cx(controls[0], controls[1])

        circuit.cp(pi/4,  controls[1], target)
        circuit.cx(controls[1], controls[2])
        circuit.cp(-pi/4, controls[2], target)
        circuit.cx(controls[0], controls[2])

        circuit.cp(pi/4,  controls[2], target)
        circuit.cx(controls[1], controls[2])
        circuit.cp(-pi/4, controls[2], target)
        circuit.cx(controls[0], controls[2])

        circuit.cp(pi/4,  controls[2], target)

---
## 3. Grover iterator

Each call to `grover_n` appends $N$ full Grover iterations to the circuit.

**Oracle for $|0010\rangle$:**  
Qubits 0, 2, 3 are 0 in the marked state, so we flip them with $X$ before and after the cccZ.
Qubit 1 is 1, so it passes through unchanged.

```
q0: ─X──●──X─    (bit 0 = 0 → flip)
q1: ────●────    (bit 1 = 1 → unchanged)
q2: ─X──●──X─    (bit 2 = 0 → flip)
q3: ─X──Z──X─    (bit 3 = 0 → flip, target of cccZ)
```

**Diffuser:**  
$H^{\otimes 4} \cdot X^{\otimes 4} \cdot \text{cccZ} \cdot X^{\otimes 4} \cdot H^{\otimes 4}$

This reflects the amplitude vector about $|s\rangle = H^{\otimes 4}|0000\rangle$.

In [ ]:
def grover_n(circuit, qr, N, n, barriers=True):
    """Append N Grover iterations to `circuit`.

    Args:
        circuit  : QuantumCircuit to append to
        qr       : QuantumRegister of size n
        N        : number of iterations
        n        : number of qubits (len(qr))
        barriers : insert barrier gates between stages (aids visualisation)
    """
    for j in range(1, N + 1):

        # ── Oracle for |0010⟩ ────────────────────────────────
        # Flip qubits that are 0 in the marked state
        circuit.x(qr[0])
        circuit.x(qr[2])
        circuit.x(qr[3])

        # 4-qubit controlled-Z (3 controls + target)
        n_controlled_Z(circuit, [qr[i] for i in range(n - 1)], qr[n - 1])

        if barriers:
            circuit.barrier()

        # Undo the flips
        circuit.x(qr[0])
        circuit.x(qr[2])
        circuit.x(qr[3])

        if barriers:
            circuit.barrier()

        # ── Diffuser (amplitude amplification) ───────────────
        circuit.h(qr)
        circuit.x(qr)

        n_controlled_Z(circuit, [qr[i] for i in range(n - 1)], qr[n - 1])

        if barriers:
            circuit.barrier()

        circuit.x(qr)
        circuit.h(qr)

        if barriers:
            circuit.barrier()

        print(f'Grover iteration {j}/{N} appended.')

---
## 4. Build the circuit

The full circuit is:
1. **Initialisation** — $H^{\otimes 4}$ creates the uniform superposition $|s\rangle$
2. **3 Grover iterations** — optimal for $n=4$ ($k = \lfloor\pi/4 \cdot \sqrt{16}\rfloor = 3$)
3. **Measurement** — collapse to the classical result

In [ ]:
n = 4           # number of qubits
N_iter = 3      # optimal Grover iterations for n=4
barriers = True

qr = QuantumRegister(n)
cr = ClassicalRegister(n)
groverCircuit = QuantumCircuit(qr, cr)

# Initialisation: uniform superposition
groverCircuit.h(qr)
if barriers:
    groverCircuit.barrier()

# Grover iterations
grover_n(groverCircuit, qr, N_iter, n, barriers=barriers)

# Measurement
groverCircuit.measure(qr, cr)

groverCircuit.draw(output='mpl')

---
## 5. Simulation

We run the circuit on Qiskit Aer's `AerSimulator` (statevector-accurate shot-based simulation).

**Expected result:** the state `0010` should have probability close to 1 after 3 iterations.  
Theoretical success probability after $k$ optimal iterations: $\sin^2((2k+1)\arcsin(1/\sqrt{N})) \approx 96\%$ for $k=3$, $N=16$.

> **Qiskit 1.x note:** The old `execute(circuit, backend, shots=shots)` one-liner is replaced by two steps:  
> 1. `transpile(circuit, backend)` — maps the logical circuit to the backend's native gate set  
> 2. `backend.run(transpiled, shots=shots)` — submit the job

In [ ]:
# Qiskit 1.x simulation — AerSimulator replaces BasicAer
backend = AerSimulator()
shots = 2**12  # 4096 shots

transpiled = transpile(groverCircuit, backend)
job = backend.run(transpiled, shots=shots)
result = job.result()

answer = result.get_counts()
print('Top outcomes:')
for state, count in sorted(answer.items(), key=lambda x: -x[1])[:5]:
    print(f'  |{state}⟩  {count:4d} shots  ({100*count/shots:.1f}%)')

plot_histogram(answer)

---
## 6. Legacy — IBM Quantum Hardware

> **Historical note.** The cells below are preserved from the original notebook (2020). They ran on `ibmq_essex`, which has since been decommissioned along with the entire first-generation IBM Q Experience fleet.
>
> The IBM Quantum provider (`IBMQ`, `least_busy`, `job_monitor`) was removed from open-source Qiskit in version 1.0. To run on real IBM hardware today, use the `qiskit-ibm-runtime` package:
> ```python
> from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
> service = QiskitRuntimeService()
> backend = service.least_busy(operational=True, simulator=False, min_num_qubits=4)
> sampler = Sampler(backend)
> job = sampler.run([transpile(groverCircuit, backend)], shots=shots)
> ```
> See the [Qiskit IBM Runtime migration guide](https://docs.quantum.ibm.com/migration-guides/qiskit-runtime).

In [ ]:
# ── LEGACY (Qiskit < 1.0) ─────────────────────────────────────────────────────
# This cell requires the old qiskit-ibmq-provider package and will not run
# with Qiskit 1.x.  Preserved for historical reference only.
#
# from qiskit import IBMQ
# from qiskit.providers.ibmq import least_busy
#
# provider = IBMQ.load_account()
# device = least_busy(provider.backends(simulator=False))
# print("Running on current least busy device: ", device)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── LEGACY (Qiskit < 1.0) ─────────────────────────────────────────────────────
# Original hardware run on ibmq_essex (decommissioned ~2021).
#
# backend = least_busy(provider.backends(
#     filters=lambda x: x.configuration().n_qubits >= 4
#                   and not x.configuration().simulator
#                   and x.status().operational == True))
# print("least busy backend: ", backend)
#
# from qiskit.tools.monitor import job_monitor
# shots = 2**12
# job = execute(groverCircuit, backend=backend, shots=shots)
# job_monitor(job, interval=2)
#
# results = job.result()
# answer  = results.get_counts(groverCircuit)
# plot_histogram(answer)
# ─────────────────────────────────────────────────────────────────────────────